---
**Script name:** 04_02_02_07_simulation_Olchard_C_random_sample.ipynb

**Purpose of script:** This script is to simulate the error in the Olchard C node and to randomly sample a subset of the data to test how the length of the dataframe alters the estimate of the correction value. The next script '04_02_02_08_simulation_Olchard_C_seed_02.py should be repeated for each value of random seed that has been selected.

**Author:** Dr Katie Allan

**Date Created:** 2022-07-10

**Email:** katie.allan@sussex.ac.uk

**License:** 

---

**Notes:** Change 'path_to_folder' to appropriate folder paths.

---

## Importing libraries that are required

In [1]:
import time
from datetime import timedelta
import pickle
import gc
import pandas as pd
import numpy as np
import os

## Functions that are used throughout the script

In [2]:
def random_subsets(df, rng, sizes):
    #create dictionary to store results
    subsets = {}
    #calculate number of rows in df
    num_rows = len(df)

    #loop to create subset of dataframes
    for size in sizes:
        #set the minimum start value
        min_start_index = 0
        #set the maximum sample value
        max_start_index = num_rows - size + 1
        #randomly select an integer to start the sample from
        start_index = rng.integers(min_start_index, max_start_index)

        #subset the data
        sample = df.iloc[start_index:start_index + size]
        #add the subset to the dictionary
        subsets[f'sample_size_{size}'] = sample.sort_values(by='time', ascending=True)

    return subsets

In [3]:
#Function to resample the time intervals of a dataframe
def resample_time_shifts(df, shift_seconds, total_shifts):
    df.set_index(['time'], inplace=True)
    offset_dfs = []

    for shift in range(total_shifts):
        offset = pd.Timedelta(seconds = shift_seconds * shift)

        resampled = df.resample('30min', origin='epoch', offset=offset)['id'].value_counts().reset_index()
        offset_df = resampled.pivot(index='time', columns='id', values='count').fillna(0)
        offset_df = offset_df.reindex(columns=all_tag_ids, fill_value = 0)

        offset_dfs.append(offset_df)
    
    return offset_dfs

In [4]:
#function to normalize each row of data
def normalize_df_rows(df):
    return df.div(df.sum(axis=1), axis=0)

## Define variables

In [5]:
#set the time and repeats for the script
shift_seconds = 10
total_shifts = 180

In [6]:
#Set the tags to filter by
all_tag_ids = ['55331961','4B551E4B', '78780778', '4C785261', '2D661E19', '33333433', '78341E61', '52616161', '662A6107', '614B2A61', '34332A07', '612A4C1E', '2D2D612A', '2D2A7852', '07071E19', '2D192D78', '1E4B074C', '4B662A33', '19557855', '2A615252', '33663378', '5561331E', '341E4B2A', '1978782D', '2D344B2D', '19616666', '551E3307', '34783378', '66667833', '4B616652', '4C610761', '33331E55', '612A3355', '4B2D664C']

In [ ]:
# File directories
input_dir = Path("../../04_01_data_wrangling/output")
output_dir = Path(".")

## Load Data

In [ ]:
#open pickle file of correct node data
with open(f'{input_dir}/correct_node_data.pickle', 'rb') as file:
    correct_dataframes = pickle.load(file)

#simulate the error in BatHouse 1 node
first_date = pd.Timestamp('2017-01-01 00:00:00')
bathouse_simulated_time_diff = correct_dataframes['node_37792f_beep_3_correct'].assign(time = correct_dataframes['node_37792f_beep_3_correct']['time'] - (correct_dataframes['node_37792f_beep_3_correct'].iloc[0, 0] - first_date))

C:\Users\katie\AppData\Local\Temp\ipykernel_9956\2289732984.py:3: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  correct_dataframes = pickle.load(file)


## Data processing and export

In [ ]:
if not os.path.isfile(f'{output_dir}/normalized_shifts_simulation_data_seed_02.pickle'):
    
    simulation_data_seed_02 = random_subsets(
        bathouse_simulated_time_diff,
        np.random.default_rng(2),
        range(100, 3100, 100)
    )
    
    with open(f'{output_dir}/simulation_data_seed_02.pickle', 'wb') as handle:
        pickle.dump(simulation_data_seed_02, handle, protocol=pickle.HIGHEST_PROTOCOL)

    #Normalize and resample subsets of data
    normalized_shifts_simulation_data_seed_02 = {key: list(map(normalize_df_rows, resample_time_shifts(df, shift_seconds, total_shifts))) for key, df in simulation_data_seed_02.items()}
    
    with open(f'{output_dir}/normalized_shifts_simulation_data_seed_02.pickle', 'wb') as handle:
        pickle.dump(normalized_shifts_simulation_data_seed_02, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
if not os.path.isfile(f'{output_dir}/normalized_shifts_simulation_data_seed_03.pickle'):
    
    simulation_data_seed_03 = random_subsets(
        bathouse_simulated_time_diff,
        np.random.default_rng(3),
        range(100, 3100, 100)
    )
    
    with open(f'{output_dir}/simulation_data_seed_03.pickle', 'wb') as handle:
        pickle.dump(simulation_data_seed_03, handle, protocol=pickle.HIGHEST_PROTOCOL)

    #Normalize and resample subsets of data
    normalized_shifts_simulation_data_seed_03 = {key: list(map(normalize_df_rows, resample_time_shifts(df, shift_seconds, total_shifts))) for key, df in simulation_data_seed_03.items()}
    
    with open(f'{output_dir}/normalized_shifts_simulation_data_seed_03.pickle', 'wb') as handle:
        pickle.dump(normalized_shifts_simulation_data_seed_03, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
if not os.path.isfile(f'{output_dir}/normalized_shifts_simulation_data_seed_18.pickle'):
    
    simulation_data_seed_18 = random_subsets(
        bathouse_simulated_time_diff,
        np.random.default_rng(18),
        range(100, 3100, 100)
    )
    
    with open(f'{output_dir}/simulation_data_seed_18.pickle', 'wb') as handle:
        pickle.dump(simulation_data_seed_18, handle, protocol=pickle.HIGHEST_PROTOCOL)

    #Normalize and resample subsets of data
    normalized_shifts_simulation_data_seed_18 = {key: list(map(normalize_df_rows, resample_time_shifts(df, shift_seconds, total_shifts))) for key, df in simulation_data_seed_18.items()}
    
    with open(f'{output_dir}/normalized_shifts_simulation_data_seed_18.pickle', 'wb') as handle:
        pickle.dump(normalized_shifts_simulation_data_seed_18, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
if not os.path.isfile(f'{output_dir}/normalized_shifts_simulation_data_seed_19.pickle'):
    
    simulation_data_seed_19 = random_subsets(
        bathouse_simulated_time_diff,
        np.random.default_rng(19),
        range(100, 3100, 100)
    )
    
    with open(f'{output_dir}/simulation_data_seed_19.pickle', 'wb') as handle:
        pickle.dump(simulation_data_seed_19, handle, protocol=pickle.HIGHEST_PROTOCOL)

    #Normalize and resample subsets of data
    normalized_shifts_simulation_data_seed_19 = {key: list(map(normalize_df_rows, resample_time_shifts(df, shift_seconds, total_shifts))) for key, df in simulation_data_seed_19.items()}
    
    with open(f'{output_dir}/normalized_shifts_simulation_data_seed_19.pickle', 'wb') as handle:
        pickle.dump(normalized_shifts_simulation_data_seed_19, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
if not os.path.isfile(f'{output_dir}/normalized_shifts_simulation_data_seed_23.pickle'):
    
    simulation_data_seed_23 = random_subsets(
        bathouse_simulated_time_diff,
        np.random.default_rng(23),
        range(100, 3100, 100)
    )
    
    with open(f'{output_dir}/simulation_data_seed_23.pickle', 'wb') as handle:
        pickle.dump(simulation_data_seed_23, handle, protocol=pickle.HIGHEST_PROTOCOL)

    #Normalize and resample subsets of data
    normalized_shifts_simulation_data_seed_23 = {key: list(map(normalize_df_rows, resample_time_shifts(df, shift_seconds, total_shifts))) for key, df in simulation_data_seed_23.items()}
    
    with open(f'{output_dir}/normalized_shifts_simulation_data_seed_23.pickle', 'wb') as handle:
        pickle.dump(normalized_shifts_simulation_data_seed_23, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
if not os.path.isfile(f'{output_dir}/normalized_shifts_simulation_data_seed_25.pickle'):
    
    simulation_data_seed_25 = random_subsets(
        bathouse_simulated_time_diff,
        np.random.default_rng(25),
        range(100, 3100, 100)
    )
    
    with open(f'{output_dir}/simulation_data_seed_25.pickle', 'wb') as handle:
        pickle.dump(simulation_data_seed_25, handle, protocol=pickle.HIGHEST_PROTOCOL)

    #Normalize and resample subsets of data
    normalized_shifts_simulation_data_seed_25 = {key: list(map(normalize_df_rows, resample_time_shifts(df, shift_seconds, total_shifts))) for key, df in simulation_data_seed_25.items()}
    
    with open(f'{output_dir}/normalized_shifts_simulation_data_seed_25.pickle', 'wb') as handle:
        pickle.dump(normalized_shifts_simulation_data_seed_25, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
if not os.path.isfile(f'{output_dir}/normalized_shifts_simulation_data_seed_30.pickle'):
    
    simulation_data_seed_30 = random_subsets(
        bathouse_simulated_time_diff,
        np.random.default_rng(30),
        range(100, 3100, 100)
    )
    
    with open(f'{output_dir}/simulation_data_seed_30.pickle', 'wb') as handle:
        pickle.dump(simulation_data_seed_30, handle, protocol=pickle.HIGHEST_PROTOCOL)

    #Normalize and resample subsets of data
    normalized_shifts_simulation_data_seed_30 = {key: list(map(normalize_df_rows, resample_time_shifts(df, shift_seconds, total_shifts))) for key, df in simulation_data_seed_30.items()}
    
    with open(f'{output_dir}/normalized_shifts_simulation_data_seed_30.pickle', 'wb') as handle:
        pickle.dump(normalized_shifts_simulation_data_seed_30, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
if not os.path.isfile(f'{output_dir}/normalized_shifts_simulation_data_seed_31.pickle'):
    
    simulation_data_seed_31 = random_subsets(
        bathouse_simulated_time_diff,
        np.random.default_rng(31),
        range(100, 3100, 100)
    )
    
    with open(f'{output_dir}/simulation_data_seed_31.pickle', 'wb') as handle:
        pickle.dump(simulation_data_seed_31, handle, protocol=pickle.HIGHEST_PROTOCOL)

    #Normalize and resample subsets of data
    normalized_shifts_simulation_data_seed_31 = {key: list(map(normalize_df_rows, resample_time_shifts(df, shift_seconds, total_shifts))) for key, df in simulation_data_seed_31.items()}
    
    with open(f'{output_dir}/normalized_shifts_simulation_data_seed_31.pickle', 'wb') as handle:
        pickle.dump(normalized_shifts_simulation_data_seed_31, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
if not os.path.isfile(f'{output_dir}/normalized_shifts_simulation_data_seed_40.pickle'):
    
    simulation_data_seed_40 = random_subsets(
        bathouse_simulated_time_diff,
        np.random.default_rng(40),
        range(100, 3100, 100)
    )
    
    with open(f'{output_dir}/simulation_data_seed_40.pickle', 'wb') as handle:
        pickle.dump(simulation_data_seed_40, handle, protocol=pickle.HIGHEST_PROTOCOL)

    #Normalize and resample subsets of data
    normalized_shifts_simulation_data_seed_40 = {key: list(map(normalize_df_rows, resample_time_shifts(df, shift_seconds, total_shifts))) for key, df in simulation_data_seed_40.items()}
    
    with open(f'{output_dir}/normalized_shifts_simulation_data_seed_40.pickle', 'wb') as handle:
        pickle.dump(normalized_shifts_simulation_data_seed_40, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
if not os.path.isfile(f'{output_dir}/normalized_shifts_simulation_data_seed_41.pickle'):
    
    simulation_data_seed_41 = random_subsets(
        bathouse_simulated_time_diff,
        np.random.default_rng(41),
        range(100, 3100, 100)
    )
    
    with open(f'{output_dir}/simulation_data_seed_41.pickle', 'wb') as handle:
        pickle.dump(simulation_data_seed_41, handle, protocol=pickle.HIGHEST_PROTOCOL)

    #Normalize and resample subsets of data
    normalized_shifts_simulation_data_seed_41 = {key: list(map(normalize_df_rows, resample_time_shifts(df, shift_seconds, total_shifts))) for key, df in simulation_data_seed_41.items()}
    
    with open(f'{output_dir}/normalized_shifts_simulation_data_seed_41.pickle', 'wb') as handle:
        pickle.dump(normalized_shifts_simulation_data_seed_41, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
if not os.path.isfile(f'{output_dir}/normalized_shifts_simulation_data_seed_56.pickle'):
    
    simulation_data_seed_56 = random_subsets(
        bathouse_simulated_time_diff,
        np.random.default_rng(56),
        range(100, 3100, 100)
    )
    
    with open(f'{output_dir}/simulation_data_seed_56.pickle', 'wb') as handle:
        pickle.dump(simulation_data_seed_56, handle, protocol=pickle.HIGHEST_PROTOCOL)

    #Normalize and resample subsets of data
    normalized_shifts_simulation_data_seed_56 = {key: list(map(normalize_df_rows, resample_time_shifts(df, shift_seconds, total_shifts))) for key, df in simulation_data_seed_56.items()}
    
    with open(f'{output_dir}/normalized_shifts_simulation_data_seed_56.pickle', 'wb') as handle:
        pickle.dump(normalized_shifts_simulation_data_seed_56, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
if not os.path.isfile(f'{output_dir}/normalized_shifts_simulation_data_seed_57.pickle'):
    
    simulation_data_seed_57 = random_subsets(
        bathouse_simulated_time_diff,
        np.random.default_rng(57),
        range(100, 3100, 100)
    )
    
    with open(f'{output_dir}/simulation_data_seed_57.pickle', 'wb') as handle:
        pickle.dump(simulation_data_seed_57, handle, protocol=pickle.HIGHEST_PROTOCOL)

    #Normalize and resample subsets of data
    normalized_shifts_simulation_data_seed_57 = {key: list(map(normalize_df_rows, resample_time_shifts(df, shift_seconds, total_shifts))) for key, df in simulation_data_seed_57.items()}
    
    with open(f'{output_dir}/normalized_shifts_simulation_data_seed_57.pickle', 'wb') as handle:
        pickle.dump(normalized_shifts_simulation_data_seed_57, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
if not os.path.isfile(f'{output_dir}/normalized_shifts_simulation_data_seed_68.pickle'):
    
    simulation_data_seed_68 = random_subsets(
        bathouse_simulated_time_diff,
        np.random.default_rng(68),
        range(100, 3100, 100)
    )
    
    with open(f'{output_dir}/simulation_data_seed_68.pickle', 'wb') as handle:
        pickle.dump(simulation_data_seed_68, handle, protocol=pickle.HIGHEST_PROTOCOL)

    #Normalize and resample subsets of data
    normalized_shifts_simulation_data_seed_68 = {key: list(map(normalize_df_rows, resample_time_shifts(df, shift_seconds, total_shifts))) for key, df in simulation_data_seed_68.items()}
    
    with open(f'{output_dir}/normalized_shifts_simulation_data_seed_68.pickle', 'wb') as handle:
        pickle.dump(normalized_shifts_simulation_data_seed_68, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
if not os.path.isfile(f'{output_dir}/normalized_shifts_simulation_data_seed_69.pickle'):
    
    simulation_data_seed_69 = random_subsets(
        bathouse_simulated_time_diff,
        np.random.default_rng(69),
        range(100, 3100, 100)
    )
    
    with open(f'{output_dir}/simulation_data_seed_69.pickle', 'wb') as handle:
        pickle.dump(simulation_data_seed_69, handle, protocol=pickle.HIGHEST_PROTOCOL)

    #Normalize and resample subsets of data
    normalized_shifts_simulation_data_seed_69 = {key: list(map(normalize_df_rows, resample_time_shifts(df, shift_seconds, total_shifts))) for key, df in simulation_data_seed_69.items()}
    
    with open(f'{output_dir}/normalized_shifts_simulation_data_seed_69.pickle', 'wb') as handle:
        pickle.dump(normalized_shifts_simulation_data_seed_69, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
if not os.path.isfile(f'{output_dir}/normalized_shifts_simulation_data_seed_73.pickle'):
    
    simulation_data_seed_73 = random_subsets(
        bathouse_simulated_time_diff,
        np.random.default_rng(73),
        range(100, 3100, 100)
    )
    
    with open(f'{output_dir}/simulation_data_seed_73.pickle', 'wb') as handle:
        pickle.dump(simulation_data_seed_73, handle, protocol=pickle.HIGHEST_PROTOCOL)

    #Normalize and resample subsets of data
    normalized_shifts_simulation_data_seed_73 = {key: list(map(normalize_df_rows, resample_time_shifts(df, shift_seconds, total_shifts))) for key, df in simulation_data_seed_73.items()}
    
    with open(f'{output_dir}/normalized_shifts_simulation_data_seed_73.pickle', 'wb') as handle:
        pickle.dump(normalized_shifts_simulation_data_seed_73, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
if not os.path.isfile(f'{output_dir}/normalized_shifts_simulation_data_seed_74.pickle'):
    
    simulation_data_seed_74 = random_subsets(
        bathouse_simulated_time_diff,
        np.random.default_rng(74),
        range(100, 3100, 100)
    )
    
    with open(f'{output_dir}/simulation_data_seed_74.pickle', 'wb') as handle:
        pickle.dump(simulation_data_seed_74, handle, protocol=pickle.HIGHEST_PROTOCOL)

    #Normalize and resample subsets of data
    normalized_shifts_simulation_data_seed_74 = {key: list(map(normalize_df_rows, resample_time_shifts(df, shift_seconds, total_shifts))) for key, df in simulation_data_seed_74.items()}
    
    with open(f'{output_dir}/normalized_shifts_simulation_data_seed_74.pickle', 'wb') as handle:
        pickle.dump(normalized_shifts_simulation_data_seed_74, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
if not os.path.isfile(f'{output_dir}/normalized_shifts_simulation_data_seed_84.pickle'):
    
    simulation_data_seed_84 = random_subsets(
        bathouse_simulated_time_diff,
        np.random.default_rng(84),
        range(100, 3100, 100)
    )
    
    with open(f'{output_dir}/simulation_data_seed_84.pickle', 'wb') as handle:
        pickle.dump(simulation_data_seed_84, handle, protocol=pickle.HIGHEST_PROTOCOL)

    #Normalize and resample subsets of data
    normalized_shifts_simulation_data_seed_84 = {key: list(map(normalize_df_rows, resample_time_shifts(df, shift_seconds, total_shifts))) for key, df in simulation_data_seed_84.items()}
    
    with open(f'{output_dir}/normalized_shifts_simulation_data_seed_84.pickle', 'wb') as handle:
        pickle.dump(normalized_shifts_simulation_data_seed_84, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
if not os.path.isfile(f'{output_dir}/normalized_shifts_simulation_data_seed_86.pickle'):
    
    simulation_data_seed_86 = random_subsets(
        bathouse_simulated_time_diff,
        np.random.default_rng(86),
        range(100, 3100, 100)
    )
    
    with open(f'{output_dir}/simulation_data_seed_86.pickle', 'wb') as handle:
        pickle.dump(simulation_data_seed_86, handle, protocol=pickle.HIGHEST_PROTOCOL)

    #Normalize and resample subsets of data
    normalized_shifts_simulation_data_seed_86 = {key: list(map(normalize_df_rows, resample_time_shifts(df, shift_seconds, total_shifts))) for key, df in simulation_data_seed_86.items()}
    
    with open(f'{output_dir}/normalized_shifts_simulation_data_seed_86.pickle', 'wb') as handle:
        pickle.dump(normalized_shifts_simulation_data_seed_86, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
if not os.path.isfile(f'{output_dir}/normalized_shifts_simulation_data_seed_96.pickle'):
    
    simulation_data_seed_96 = random_subsets(
        bathouse_simulated_time_diff,
        np.random.default_rng(96),
        range(100, 3100, 100)
    )
    
    with open(f'{output_dir}/simulation_data_seed_96.pickle', 'wb') as handle:
        pickle.dump(simulation_data_seed_96, handle, protocol=pickle.HIGHEST_PROTOCOL)

    #Normalize and resample subsets of data
    normalized_shifts_simulation_data_seed_96 = {key: list(map(normalize_df_rows, resample_time_shifts(df, shift_seconds, total_shifts))) for key, df in simulation_data_seed_96.items()}
    
    with open(f'{output_dir}/normalized_shifts_simulation_data_seed_96.pickle', 'wb') as handle:
        pickle.dump(normalized_shifts_simulation_data_seed_96, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
if not os.path.isfile(f'{output_dir}/normalized_shifts_simulation_data_seed_97.pickle'):
    
    simulation_data_seed_97 = random_subsets(
        bathouse_simulated_time_diff,
        np.random.default_rng(97),
        range(100, 3100, 100)
    )
    
    with open(f'{output_dir}/simulation_data_seed_97.pickle', 'wb') as handle:
        pickle.dump(simulation_data_seed_97, handle, protocol=pickle.HIGHEST_PROTOCOL)

    #Normalize and resample subsets of data
    normalized_shifts_simulation_data_seed_97 = {key: list(map(normalize_df_rows, resample_time_shifts(df, shift_seconds, total_shifts))) for key, df in simulation_data_seed_97.items()}
    
    with open(f'{output_dir}/normalized_shifts_simulation_data_seed_97.pickle', 'wb') as handle:
        pickle.dump(normalized_shifts_simulation_data_seed_97, handle, protocol=pickle.HIGHEST_PROTOCOL)